In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/02_silver_cleaning/00_common_functions

In [0]:
df = spark.read.table("hive_metastore.bronze.bronze_event_log")
display(df.limit(5))
df.printSchema()

In [0]:
df_event = df.filter(col("EVDESC").contains("Alto"))

# Show the resulting DataFrame
df_event.display()

In [0]:
alarme_df = df.filter(
    lower(col("EVDESC")).like("%falha%")      # contains “falha”, ignoring case
)

display(alarme_df)

In [0]:
df = parse_ts(df, "EVDATE")

display(df)

In [0]:
df_test = df  # or: spark.table("catalog.schema.event_log")

In [0]:
id_cols = [
    "ID_TIPOEVENTO1",
    "ID_TIPOEVENTO2",
    "ID_TIPOEVENTO3",
    "ID_TIPOEVENTO4",
    "ID_TIPOEVENTO5",
]

df_sel = df_test.select(*id_cols, "DATE", "EVDESC")

In [0]:
w = Window.partitionBy(*id_cols).orderBy(F.col("DATE").desc_nulls_last())

df_top2 = (
    df_sel
    .filter(F.col("EVDESC").isNotNull())
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") <= 2)
)

In [0]:
result = (
    df_top2
    .groupBy(*id_cols)
    .agg(
        F.sort_array(F.collect_list(F.struct("rn", "DATE", "EVDESC"))).alias("samples")
    )
    .withColumn("evdesc_examples", F.expr("transform(samples, x -> x.EVDESC)"))
    .withColumn("evdate_examples", F.expr("transform(samples, x -> x.DATE)"))
    .drop("samples")
)

display(result)

In [0]:
id_cols = [
    "ID_TIPOEVENTO1",
    "ID_TIPOEVENTO2",
    "ID_TIPOEVENTO3",
    "ID_TIPOEVENTO4",
    "ID_TIPOEVENTO5",
]

counts = (
    df_sel
    .groupBy(*id_cols)
    .agg(F.count("*").alias("n_events"))
)

In [0]:
result_with_count = (
    result
    .join(counts, on=id_cols, how="left")
)

display(result_with_count)

In [0]:
df_EL = df.select("TAG1", "EVDESC", "DATE")

In [0]:
dbutils.data.summarize(df_EL)

897M de registos

min: 2023-07-04T10:25:19Z

max: 2024-07-03T00:00:38Z

Pivoting the ID column

In [0]:
df_p = df_EL.withColumn("ID_prefix", substring(col("TAG1"), 1, 6))

In [0]:
interval_s = 15 * 60  # 900

df_p = df_p.withColumn(
    "DATE_15M",
    F.to_timestamp(
        F.from_unixtime(
            (F.round(F.unix_timestamp(col("DATE")) / interval_s) * interval_s).cast("long")
        )
    )
)

display(df_p.select("DATE", "DATE_15M").limit(20))


In [0]:
df_p = (
    df_p
    .drop("DATE")
    .withColumnRenamed("DATE_15M", "DATE")
)

In [0]:
display(
    df_p.select(
        ((F.unix_timestamp(col("DATE")) % 900)).alias("offset_s")
    )
    .groupBy("offset_s")
    .count()
    .orderBy("offset_s")
)


In [0]:
display(df_p)

In [0]:
events_per_bucket = (
    df_p
    .groupBy("ID_prefix", "DATE")
    .agg(F.count(F.lit(1)).alias("events_15m_cnt"))
)

display(events_per_bucket.orderBy("ID_prefix", "DATE"))

In [0]:
df_with_cnt = (
    df_p
    .join(events_per_bucket, on=["ID_prefix", "DATE"], how="left")
)

display(df_with_cnt)

In [0]:
expected_len = len("LD6D4A-URT-URFCM")  # 16

df_clean = (
    df_with_cnt
    .withColumn("TAG1_trim", F.trim(F.col("TAG1")))
    .filter(F.col("TAG1_trim").isNotNull())
    .filter(F.length(F.col("TAG1_trim")) == expected_len)
    .drop("TAG1_trim")
)

display(df_clean)

In [0]:
display(
    df_clean.withColumn("tag1_len", F.length(F.trim(F.col("TAG1"))))
      .groupBy("tag1_len")
      .count()
      .orderBy(F.col("count").desc())
)

In [0]:
target_catalog = "hive_metastore"     # change this
target_schema = "silver"
target_table = "silver_event_log"  # change this

full_name = f"{target_catalog}.{target_schema}.{target_table}"


In [0]:
(
    df_with_cnt.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_name)
)

display(spark.table(full_name).limit(20))


In [0]:
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")